In [1]:
import os
from framework.researcher import find_researcher_ids, get_papers_by_researcher_id, keep_target_author
from framework.pindex import compute_pindex
os.makedirs("output", exist_ok=True)

In [18]:
#Step 1: Find the unique research ID (There can be many researchers who use the same name)
#FIRST_NAME = "Michel Tuan"
#LAST_NAME = "Pham"

FIRST_NAME = "Kamel"
LAST_NAME = "Jedidi"

rid_df, rid_counter = find_researcher_ids(FIRST_NAME, LAST_NAME)

print("Candidate researcherId records:")
print(rid_df)

print("\nresearcherId frequencies:")
print(rid_counter.most_common())

if len(rid_counter) == 0:
    raise ValueError("No researcherId was found.")

TARGET_RID = rid_counter.most_common(1)[0][0]
#TARGET_RID = "CFP-4440-2022"
print("\nRecommended TARGET_RID:", TARGET_RID)

Candidate researcherId records:
                    uid                                              title  \
0   WOS:000208062500004  On Estimating Finite Mixtures of Multivariate ...   
1   WOS:000249102200003  Modeling preference evolution in discrete choi...   
2   WOS:000237884000007  Identifying sources of heterogeneity for empir...   
3   WOS:000242835400004  A multibrand concept-testing methodology for n...   
4   WOS:000248660600008  Representation and inference of lexicographic ...   
5   WOS:000254588200006           A conjoint approach to multipart pricing   
6   WOS:000258604000003  Inferring latent class lexicographic rules fro...   
7   WOS:000301444800004  Willingness to pay: measurement and managerial...   
8   WOS:000272826600011  A Conjoint Approach for Consumer- and Firm-Lev...   
9   WOS:000282749800009  Dynamic Allocation of Pharmaceutical Detailing...   
10  WOS:000295953500005  The Impact of Tariff Structure on Customer Ret...   
11  WOS:000302386200009         

In [19]:
# Step 2: Make the list of the paper using own research ID
papers_final_df = get_papers_by_researcher_id(TARGET_RID)
papers_final_df = keep_target_author(papers_final_df, FIRST_NAME, LAST_NAME)

print("\nFinal paper count:", len(papers_final_df))
print(papers_final_df.head(20))

save_cols = [
    "uid", "title", "journal", "year", "times_cited",
    "doi", "document_types", "source_types", "authors"
]

papers_final_df[save_cols].to_csv(
    "output/papers_final.csv",
    index=False,
    encoding="utf-8-sig"
)


Final paper count: 21
                    uid                                              title  \
0   WOS:000208062500004  On Estimating Finite Mixtures of Multivariate ...   
1   WOS:000249102200003  Modeling preference evolution in discrete choi...   
2   WOS:000237884000007  Identifying sources of heterogeneity for empir...   
3   WOS:000242835400004  A multibrand concept-testing methodology for n...   
4   WOS:000248660600008  Representation and inference of lexicographic ...   
5   WOS:000254588200006           A conjoint approach to multipart pricing   
6   WOS:000258604000003  Inferring latent class lexicographic rules fro...   
7   WOS:000301444800004  Willingness to pay: measurement and managerial...   
8   WOS:000272826600011  A Conjoint Approach for Consumer- and Firm-Lev...   
9   WOS:000282749800009  Dynamic Allocation of Pharmaceutical Detailing...   
10  WOS:000295953500005  The Impact of Tariff Structure on Customer Ret...   
11  WOS:000302386200009             A Con

In [20]:
# Step 2.5: Select rows to use for the p-index
# Use all rows:
# SELECTED_ROWS = "all"

# Use only selected rows by row number shown in papers_final_df:
# SELECTED_ROWS = [0, 2, 5]

SELECTED_ROWS = "all"
#SELECTED_ROWS = [0, 2, 5]

if SELECTED_ROWS == "all" or SELECTED_ROWS is None:
    papers_selected_df = papers_final_df.copy()
else:
    papers_selected_df = papers_final_df.iloc[SELECTED_ROWS].copy()

papers_selected_df = papers_selected_df.reset_index(drop=True)


In [21]:
# Step 3: Calculate p index

papers_with_pr, pindex = compute_pindex(papers_selected_df)

print("\np-index:", pindex)
print(papers_with_pr[[
    "title", "journal", "year", "times_cited", "cell_size", "pr"
]].head(20))

papers_with_pr.to_csv(
    "output/papers_with_pr.csv",
    index=False,
    encoding="utf-8-sig"
)


p-index: 0.42492832697793487
                                                title  \
0   On Estimating Finite Mixtures of Multivariate ...   
1   Modeling preference evolution in discrete choi...   
2   Identifying sources of heterogeneity for empir...   
3   A multibrand concept-testing methodology for n...   
4   Representation and inference of lexicographic ...   
5            A conjoint approach to multipart pricing   
6   Inferring latent class lexicographic rules fro...   
7   Willingness to pay: measurement and managerial...   
8   A Conjoint Approach for Consumer- and Firm-Lev...   
9   Dynamic Allocation of Pharmaceutical Detailing...   
10  The Impact of Tariff Structure on Customer Ret...   
11             A Conjoint Model of Quantity Discounts   
12  Functional and experiential routes to persuasi...   
13  How to Advertise and Build Brand Knowledge Glo...   
14  Social Contagion and Customer Adoption of New ...   
15            Error Theory for Elimination by Aspects   
1